In [ ]:
# 运行此代码单元格来挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# 成功后 Google Drive内容将可以在以下路径访问：
# '/content/drive/My Drive/'
# 假如文件夹名为 'my_project_data' 并且存储在Google Drive的根目录下
# 在Colab中就可以通过 '/content/drive/My Drive/my_project_data/' 来访问它。
drive_path = '/content/drive/My Drive/自选题' # 定义想要切换到的文件夹路径

if os.path.exists(drive_path):
    os.chdir(drive_path)
    print(f"当前工作目录已切换至: {os.getcwd()}")
else:
    print(f"错误: 路径不存在 {drive_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
当前工作目录已切换至: /content/drive/My Drive/自选题


# Experiment: EMCAD Ablation And Failure Analysis On ETIS

这一册基于真实 ETIS 数据运行消融版本，并与 `01` 的 baseline 结果做统一对比。


## 加载基础工具

这里只使用环境与结果保存工具，真实消融逻辑都写在本 notebook 内。


In [ ]:
from pathlib import Path
import json
import os
import random
import sys

from scripts.project_utils import (
    ARTIFACTS,
    DATA_ROOT,
    ETIS_ROOT,
    PVT_PRETRAINED_ROOT,
    ensure_project_dirs,
    get_default_project_config,
    load_project_config,
    print_env_summary,
    save_json,
    set_seed,
    try_import_torch,
)

ensure_project_dirs()
set_seed()
torch = try_import_torch()
ENV_SUMMARY = print_env_summary(torch)
ROOT = Path.cwd().resolve()


{
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "root": "/content/drive/My Drive/自选题",
  "data_root": "/content/drive/My Drive/自选题/data",
  "etis_root": "/content/drive/My Drive/自选题/data/ETIS",
  "pvt_pretrained_root": "/content/drive/My Drive/自选题/data/pvt_pretrained_pth",
  "torch_installed": true,
  "cuda_available": true
}


## 读取共享实验配置

这里直接读取 `00` 已保存的统一配置，继续沿用 `01` 的 ETIS 路径、训练参数和统一测试样本。


In [ ]:
PROJECT_CONFIG = load_project_config()
PROJECT_CONFIG


{'dataset': 'ETIS',
 'task': 'Polyp Segmentation',
 'paper_repo': 'https://github.com/SLDGroup/EMCAD',
 'baseline_repos': {'Swin-Unet': 'https://github.com/HuCaoFighting/Swin-Unet',
  'U-Net': 'https://github.com/milesial/Pytorch-UNet'},
 'emcad_scale': 'PVT-EMCAD-B0',
 'metrics': ['Dice'],
 'fixed_visual_sample': '100.png',
 'etis_paths': {'root': '/content/drive/My Drive/自选题/data/ETIS',
  'train_images': '/content/drive/My Drive/自选题/data/ETIS/train/images',
  'train_masks': '/content/drive/My Drive/自选题/data/ETIS/train/masks',
  'val_images': '/content/drive/My Drive/自选题/data/ETIS/val/images',
  'val_masks': '/content/drive/My Drive/自选题/data/ETIS/val/masks',
  'test_images': '/content/drive/My Drive/自选题/data/ETIS/test/images',
  'test_masks': '/content/drive/My Drive/自选题/data/ETIS/test/masks',
  'train_list': '/content/drive/My Drive/自选题/data/ETIS/train_list_etis.txt',
  'val_list': '/content/drive/My Drive/自选题/data/ETIS/val_list_etis.txt',
  'test_list': '/content/drive/My Drive/自选题/

## 读取 EMCAD baseline 结果

baseline 结果固定来自 `01_emcad_full_training.ipynb`。


In [ ]:
baseline_result_path = ARTIFACTS / "records" / "emcad_etis_results.json"
baseline_reference = json.loads(baseline_result_path.read_text(encoding="utf-8")) if baseline_result_path.exists() else {}
baseline_reference.get("test_summary", {})


{'loss': 0.6782, 'dice': 0.8787}

## 复用 baseline 数据与组件

这里直接复用 `01` 的 ETIS 数据流和 EMCAD 结构组件，但不再重复整段 baseline notebook 内容。


In [ ]:
import json
from pathlib import Path

baseline_nb = json.loads(Path("01_emcad_full_training.ipynb").read_text(encoding="utf-8"))
for idx in [3, 5, 7, 9]:
    exec("".join(baseline_nb["cells"][idx]["source"]), globals())


{
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "root": "/content/drive/My Drive/自选题",
  "data_root": "/content/drive/My Drive/自选题/data",
  "etis_root": "/content/drive/My Drive/自选题/data/ETIS",
  "pvt_pretrained_root": "/content/drive/My Drive/自选题/data/pvt_pretrained_pth",
  "torch_installed": true,
  "cuda_available": true
}


## 定义消融模型

这里不复制 baseline 全结构，只定义消融后发生变化的地方：多尺度卷积核由 `(1, 3, 5)` 变为 `(3,)`。


In [ ]:
class EMCADAblationModel(EMCADBaseline):
    def __init__(self):
        super().__init__(kernel_sizes=(3,))


## 说明 baseline 与 ablation 的结构差异

这一段明确指出消融改动点及其分析目标。


In [ ]:
ablation_design = {
    "baseline_source": "01_emcad_full_training.ipynb",
    "baseline_summary": "EMCAD baseline uses MSCAM with kernel sizes (1, 3, 5).",
    "ablation_change": "Replace multi-scale MSCAM kernels with a single scale kernel set (3,).",
    "analysis_goal": "Measure how much explicit multi-scale decoding contributes on the ETIS split.",
}
ablation_design


{'baseline_source': '01_emcad_full_training.ipynb',
 'baseline_summary': 'EMCAD baseline uses MSCAM with kernel sizes (1, 3, 5).',
 'ablation_change': 'Replace multi-scale MSCAM kernels with a single scale kernel set (3,).',
 'analysis_goal': 'Measure how much explicit multi-scale decoding contributes on the ETIS split.'}

## 在 ETIS 上训练消融模型

这里继续使用同一数据和同一训练流程，对消融版本完成完整训练与测试。


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = PROJECT_CONFIG["train"]
train_loader, val_loader, test_loader = build_dataloaders(cfg)
weight_path = Path(PROJECT_CONFIG["pvt_pretrained_path"])
assert weight_path.exists(), f"未找到预训练权重: {weight_path}"

ablation_model = EMCADAblationModel().to(device)
ablation_model.load_pretrained_backbone(weight_path)
ablation_history, ablation_best_val_dice = train_model(ablation_model, train_loader, val_loader, cfg, device=device)
ablation_val_summary, ablation_val_rows = evaluate_loader(ablation_model, val_loader, device=device)
ablation_test_summary, ablation_test_rows = evaluate_loader(ablation_model, test_loader, device=device)
ablation_test_summary


{'loaded_weight_path': '/content/drive/My Drive/自选题/data/pvt_pretrained_pth/pvt_v2_b0.pth', 'missing_keys': 0, 'unexpected_keys': 2}
{'epoch': 1, 'train_loss': 1.5845, 'train_dice': 0.1388, 'val_loss': 1.6572, 'val_dice': 0.0852}
{'epoch': 2, 'train_loss': 1.4659, 'train_dice': 0.2686, 'val_loss': 1.5967, 'val_dice': 0.1693}
{'epoch': 3, 'train_loss': 1.3748, 'train_dice': 0.4235, 'val_loss': 1.4159, 'val_dice': 0.4539}
{'epoch': 4, 'train_loss': 1.3323, 'train_dice': 0.5218, 'val_loss': 1.3281, 'val_dice': 0.553}
{'epoch': 5, 'train_loss': 1.3139, 'train_dice': 0.5851, 'val_loss': 1.3016, 'val_dice': 0.579}
{'epoch': 6, 'train_loss': 1.2856, 'train_dice': 0.6269, 'val_loss': 1.2764, 'val_dice': 0.5973}
{'epoch': 7, 'train_loss': 1.2768, 'train_dice': 0.6459, 'val_loss': 1.2624, 'val_dice': 0.5617}
{'epoch': 8, 'train_loss': 1.2615, 'train_dice': 0.6863, 'val_loss': 1.2551, 'val_dice': 0.6781}
{'epoch': 9, 'train_loss': 1.2448, 'train_dice': 0.7013, 'val_loss': 1.2348, 'val_dice': 0.60

## 导出消融模型的统一测试样本结果

这里继续使用与 baseline 相同的测试样本，方便直接观察多尺度模块简化后带来的差异。


In [ ]:
ablation_history_path = ARTIFACTS / "figures" / "emcad_ablation_history.png"
saved_ablation_history = save_history_figure(ablation_history, ablation_history_path)
ablation_visual_path = ARTIFACTS / "figures" / "emcad_ablation_visual_sample.png"
saved_ablation_visual = export_visualization(
    ablation_model,
    default_visual_sample(),
    cfg["image_size"],
    ablation_visual_path,
    device=device,
)
saved_ablation_visual


'/content/drive/My Drive/自选题/artifacts/figures/emcad_ablation_visual_sample.png'

## 生成 baseline 与 ablation 对比

这里把 `01` 的 baseline 结果和当前消融结果放在同一张表里，突出结构差异带来的影响。


In [ ]:
comparison = []
if baseline_reference:
    comparison.append(
        {
            "variant": "EMCAD baseline",
            "best_val_dice": baseline_reference["best_val_dice"],
            "test_summary": baseline_reference["test_summary"],
            "difference": "Reference from 01 with MSCAM kernels (1, 3, 5)",
            "visual_path": baseline_reference["visual_path"],
        }
    )

comparison.append(
    {
        "variant": "Single-scale MSCAM ablation",
        "best_val_dice": ablation_best_val_dice,
        "test_summary": ablation_test_summary,
        "difference": "MSCAM kernels changed to (3,)",
        "visual_path": saved_ablation_visual,
    }
)
comparison


[{'variant': 'EMCAD baseline',
  'best_val_dice': 0.8128,
  'test_summary': {'loss': 0.6782, 'dice': 0.8787},
  'difference': 'Reference from 01 with MSCAM kernels (1, 3, 5)',
  'visual_path': '/content/drive/My Drive/自选题/artifacts/figures/emcad_visual_sample.png'},
 {'variant': 'Single-scale MSCAM ablation',
  'best_val_dice': 0.8055,
  'test_summary': {'loss': 0.7254, 'dice': 0.8382},
  'difference': 'MSCAM kernels changed to (3,)',
  'visual_path': '/content/drive/My Drive/自选题/artifacts/figures/emcad_ablation_visual_sample.png'}]

## 写入失败分析入口

这里把 ETIS 环境下的失败分析入口固定下来，后续可直接结合同一测试样本的分割图做肉眼分析。


In [ ]:
ablation_checkpoint_path = ARTIFACTS / "checkpoints" / "emcad_ablation_etis_best.pth"
torch.save(ablation_model.state_dict(), ablation_checkpoint_path)

failure_analysis = {
    "baseline_source": "01_emcad_full_training.ipynb",
    "visual_sample": default_visual_sample(),
    "baseline_visual_path": baseline_reference.get("visual_path"),
    "ablation_visual_path": saved_ablation_visual,
    "failure_modes": [
        {
            "name": "small polyp under-segmentation",
            "hypothesis": "single-scale decoding weakens multi-scale context aggregation",
            "evidence_placeholder": "attach per-sample Dice and qualitative figures here",
        },
        {
            "name": "boundary leakage",
            "hypothesis": "reduced receptive field hurts ambiguous polyp boundaries",
            "evidence_placeholder": "attach boundary comparison figures here",
        },
    ],
}

save_json(
    {
        "dataset": "ETIS",
        "pretrained_path": str(weight_path),
        "ablation_design": ablation_design,
        "history": ablation_history,
        "best_val_dice": ablation_best_val_dice,
        "val_summary": ablation_val_summary,
        "test_summary": ablation_test_summary,
        "val_rows": ablation_val_rows,
        "test_rows": ablation_test_rows,
        "history_figure_path": saved_ablation_history,
        "visual_sample": default_visual_sample(),
        "visual_path": saved_ablation_visual,
        "checkpoint_path": str(ablation_checkpoint_path),
        "comparison": comparison,
        "failure_analysis": failure_analysis,
    },
    ARTIFACTS / "records" / "ablation_and_failure_analysis_etis.json",
)
